In [ ]:
library(Seurat)
library(readr)
library(dplyr)
library(ggplot2)
library(pheatmap)
library(tidyr) 
library(patchwork)
library(tidyverse)
library(ggrepel)

In [ ]:
## 分群可视化（配色与前面相同）

In [ ]:
data <- readRDS("results/tacco/tacco_output_file")
data

In [ ]:
cluster_colours <- c(
  EX_ADRA1A_Vat1l = "#E69F00",   # Nature Communications橙（保留原活力色）[2,8](@ref)
  EX_BDNF = "#2D5F9E",           # 加深为Nature经典深蓝（原#56B4E9 → 提升饱和度与深度）[2,7](@ref)
  EX_C1QL3 = "#006D4D",          # 加深为Science深绿（原#009E73 → 增强学术感）[2,3](@ref)
  EX_ERG = "#004B87",            # 加深为Nature主刊深蓝（原#0072B2 → 更沉稳）[2,8](@ref)
  EX_FOXP2 = "#D55E00",          # Immunity橙红（保留高对比度警示色）[2](@ref)
  EX_HTR7_GNAL = "#CC79A7",      # Nature Methods紫红（保留柔和对比）[2](@ref)
  EX_IFITM3_CLDN5 = "#332288",   # Nature Neuroscience深紫（保留冷色调）[2](@ref)
  EX_ITGA4_TPBG = "#88CCEE",     # Science Advances浅蓝（保留清新辅助色）[2](@ref)
  EX_NTNG1 = "#7E1F86",          # 加深为Cell深紫（原#AA4499 → 降低明度提升质感）[2,4](@ref)
  EX_Ntng2 = "#117733",          # Nature Medicine深绿（保留自然风格）[2](@ref)
  EX_Nxph4_CPLX3 = "#661100",    # Cell Reports棕红（保留暖深色）[2](@ref)
  EX_SLC1A3_FGFR3 = "#999933",   # Science Signaling橄榄绿（保留中性调和）[2](@ref)
  EX_TSHZ2_VWC2L = "#DDCC77",    # Nature Genetics米黄（保留浅暖色平衡）[2](@ref)
  EX_env_TMEM144 = "#44AA99"      # Neuron蓝绿（保留科技感点缀色）[2](@ref)
)

In [ ]:
labels <- names(cluster_colours)
default_colour <- "#FFFFFF"  # 使用白色背景
output_dir <- "results/figures/figure_panel_file"

if (!dir.exists(output_dir)) {
  dir.create(output_dir, recursive = TRUE, mode = "0775")
  cat("makdir", output_dir, "\n")
}

# 循环绘制每个标签的空间分布图
for (label in labels) {
  if (label %in% unique(data$pred_celltype)) {
    # 创建一个因子变量，用于分组
    data$plot_group <- ifelse(data$pred_celltype == label, label, "Other")
    data$plot_group <- factor(data$plot_group, levels = c(label, "Other"))
    
    # 获取空间坐标
    spatial_coords <- data@meta.data[, c("x", "y")]
    
    # 绘制空间分布图
    p <- ggplot(data = spatial_coords, aes(x = x, y = y, color = data$plot_group)) +
      geom_point(aes(color = data$plot_group, alpha = data$plot_group), size = 1) +
      scale_color_manual(values = c(cluster_colours[label], default_colour)) +
      scale_alpha_manual(values = c(1, 0.2)) +  # 设置透明度，label为1（不透明），Other为0.2（较透明）
      theme_void() + 
      theme(
        panel.background = element_blank(),    # 移除绘图区域背景色
        plot.background = element_blank(),     # 移除整个图形背景色
        legend.position = "none"               # 移除图例
      )
    
    # 构建带路径的文件名
    png_filename <- file.path(output_dir, paste0(label, "_spatial_distribution.png"))
    pdf_filename <- file.path(output_dir, paste0(label, "_spatial_distribution.pdf"))
    
    # 保存为PNG和PDF文件
    ggsave(filename = png_filename, plot = p, device = "png", width = 8, height = 10, dpi = 300)
    ggsave(filename = pdf_filename, plot = p, device = "pdf", width = 8, height = 10)
    cat("Saved", png_filename, "and", pdf_filename, "\n")
  } else {
    cat("anno", label, "not find, skipped!\n")
  }
}
cat("ALL SAVED TO:", output_dir, "\n")

In [ ]:
## 绘制指定基因小提琴图

In [ ]:
# 定义基因列表
genes <- c("NTNG1", "RORB")

for (gene in genes) {
  # 创建无标注小提琴图
  vlnplot <- VlnPlot(
    data, 
    features = gene,
    group.by = "annotation",
    pt.size = 0  # 移除散点
  ) +
  # 核心优化点（移除所有文字和箱线图）
  theme(
    axis.text = element_text(size = 10),  # 缩小坐标轴刻度文字大小
    axis.title = element_blank(),       # 移除XY轴标题
    axis.ticks = element_blank(),       # 隐藏刻度线
    plot.title = element_blank(),       # 删除基因标题
    panel.grid.major = element_blank(), # 清除网格线
    panel.grid.minor = element_blank(),
    legend.text = element_text(size = 10),  # 缩小图例文字大小
    plot.margin = margin(10, 10, 10, 30)  # 增加左边的空白
  ) +
  scale_fill_manual(values = cluster_colours)  # 保留分组配色
  
  # 显示图形
  print(vlnplot)
  
  # 保存图像
  output_png <- sprintf("results/figures/figure_panel_file", gene)
  output_pdf <- sprintf("results/figures/figure_panel_file", gene)
  
  ggsave(output_png, plot = vlnplot, width = 10, height = 6, units = "in", dpi = 300)
  ggsave(output_pdf, plot = vlnplot, width = 10, height = 6, units = "in")
  
  cat("Cleaned VlnPlot for", gene, "savedata/local_or_external_file  PNG:", output_png, "\n  PDF:", output_pdf, "\n")
}

In [ ]:
## 经典L4 marker气泡图

In [ ]:
markers_L4 <- c('RORB','NTNG1','CUX1','THSD7A','NECAB1','UNC5D','OSTN')#,'SLC17A7'
markers_L4

In [ ]:
# 绘制点图
dotplot <- DotPlot(data, features = markers_L4, group.by = "annotation", 
                   dot.scale = 6,cols=c("#1F66AC","#B2192B"),col.min = -1,col.max = 2) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  theme(plot.title = element_blank()) + # 去除标题
  theme(axis.title.x = element_blank(), # 去除横坐标轴标题
        axis.title.y = element_blank()) # 去除纵坐标轴标题

# 保存图像
ggsave("results/figures/figure_panel_file", plot = dotplot, width = 6, height = 8)
ggsave("results/figures/figure_panel_file", plot = dotplot, width = 6, height = 8)